# Step 3 Manifest
Which stays are eligible at each site: admitted 2013 or later, in PICANet, with real vitals. One row per stay. This rebuild runs the three manifests again and checks every waterfall count against the baseline recorded before the step 1 and 2 rebuild (`BASELINE_manifest_waterfalls.md`). If a count differs, the cell stops with an error and nothing downstream should be trusted until it is explained. Counts only.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import manifest as M
import diagnostics as D

In [ ]:
base = Path.home() / "Documents/DS217/users/jormond"
rbh_dir, smh_dir, gosh_dir = base / "RBH", base / "SMH", base / "GOSH"
open_picu_dir = base / "open_picu"
reports_dir = open_picu_dir / "derived" / "reports"

rbh_m, rbh_w = M.rbh_manifest(rbh_dir)
smh_m, smh_w = M.smh_manifest(smh_dir)
gosh_m, gosh_w = M.gosh_manifest(gosh_dir)
waterfalls = {"RBH": pd.DataFrame(rbh_w), "SMH": pd.DataFrame(smh_w), "GOSH": pd.DataFrame(gosh_w)}
D.show(waterfalls)

## 1. Same as the baseline?
Each count is compared with the recorded baseline: RBH 14,792 / 5,112 / 4,789 / 4,765; SMH 4,047 / 3,709 / 3,709 / 3,699; GOSH 10,387 / 9,820 / 9,820 / 9,820 / 9,671. `check_waterfall` raises an error if any step differs, so a clean run means the rebuild changed nothing downstream.

In [ ]:
checks = {site: M.check_waterfall(w, site) for site, w in (("RBH", rbh_w), ("SMH", smh_w), ("GOSH", gosh_w))}
D.show(checks)

## 2. Where the manifests leave us
Stays per admission year, patients and repeat stays, and a fingerprint of each manifest (the number of stays and a short hash of the sorted stay keys). The fingerprint is recorded so a later run can show it produced the same stays, not just the same count.

In [ ]:
describe = {"RBH": M.describe(rbh_m, "PATIENT_RK"), "SMH": M.describe(smh_m, "UniquePtID"), "GOSH": M.describe(gosh_m, "project_id")}
fingerprints = pd.DataFrame({"RBH": M.fingerprint(rbh_m, "stay_key"),
                             "SMH": M.fingerprint(smh_m, "encounterId"),
                             "GOSH": M.fingerprint(gosh_m, "stay_id")}).T
D.show({"Stays per admission year": pd.concat(describe, axis=1), "Fingerprints": fingerprints})

## 3. Save

In [ ]:
D.write_report({"Waterfalls": waterfalls, "Check against the baseline": checks,
                "Stays per admission year": pd.concat(describe, axis=1), "Fingerprints": fingerprints},
               reports_dir / f"Manifest_{pd.Timestamp.today().date()}.md", title="Manifest")

## 3b. Save the manifests
Assembly reads the manifests from `derived/`, so this notebook writes them. They hold one row per eligible stay with the original keys and exact admission times, so they are row-level data: `derived/`, git-ignored, inside the SDE. The next section reads them back and checks them.

In [ ]:
derived = open_picu_dir / "derived"
rbh_m.to_csv(derived / "RBH_manifest.csv", index=False)
smh_m.to_csv(derived / "SMH_manifest.csv", index=False)
gosh_m.to_csv(derived / "GOSH_manifest.csv", index=False)

## 4. Are the saved manifests the same stays?
The assembly notebook does not run the manifest code. It reads three saved files: `derived/RBH_manifest.csv`, `derived/SMH_manifest.csv` and `derived/GOSH_manifest.csv`. Checks 1 to 3 above only show that the code, run today, gives the baseline stays. They do not show that the files assembly actually read hold those same stays.

This cell reads the three saved files back and computes the same fingerprint for each: the number of stays and a short hash of the sorted stay keys (`stay_key` for RBH, `encounterId` for SMH, `stay_id` for GOSH). The hash changes if any stay is added, removed or replaced, even when the count stays the same. The hashes must equal the ones recorded in `BASELINE_manifest_waterfalls.md` (RBH `4e40e77c2117`, SMH `19514737e1c0`, GOSH `fa73768a3bb3`). If they do, the assembled tables were built from the same stays as today's manifests. The hash cannot be turned back into a stay key, so it is safe to keep in a report. Counts only.

In [ ]:
derived = open_picu_dir / "derived"
saved = {"RBH": (pd.read_csv(derived / "RBH_manifest.csv", dtype=str), "stay_key"),
         "SMH": (pd.read_csv(derived / "SMH_manifest.csv", dtype=str), "encounterId"),
         "GOSH": (pd.read_csv(derived / "GOSH_manifest.csv", dtype=str), "stay_id")}
pd.DataFrame({site: M.fingerprint(df, key) for site, (df, key) in saved.items()}).T

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import inventory as I
import manifest as M

In [ ]:
(open_picu_dir/"derived").mkdir(exist_ok=True)
(open_picu_dir/"reports").mkdir(exist_ok=True)

# SMH

In [ ]:
smh_dir = Path("/home/jormond/Documents/DS217/users/jormond/SMH")
smh_m, smh_w = M.smh_manifest(smh_dir)
M.print_waterfall(smh_w, "SMH")
smh_m.to_csv(open_picu_dir/"derived"/"SMH_manifest.csv", index=False)

# RBH

In [ ]:
rbh_dir = Path("/home/jormond/Documents/DS217/users/jormond/RBH")
rbh_m, rbh_w = M.rbh_manifest(rbh_dir)
M.print_waterfall(rbh_w, "RBH")
rbh_m.to_csv(open_picu_dir/"derived"/"RBH_manifest.csv", index=False)

# GOSH

In [ ]:
gosh_dir = Path("/home/jormond/Documents/DS217/users/jormond/GOSH")
gosh_m, gosh_w = M.gosh_manifest(gosh_dir)
M.print_waterfall(gosh_w, "GOSH")
gosh_m.to_csv(open_picu_dir/"derived"/"GOSH_manifest.csv", index=False)

# Years

In [ ]:
rbh_years = M.describe(rbh_m, "PATIENT_RK")
smh_years = M.describe(smh_m, "UniquePtID")
gosh_years = M.describe(gosh_m, "project_id")
pd.concat({"RBH": rbh_years["stays"], "SMH": smh_years["stays"], "GOSH": gosh_years["stays"]}, axis=1)

### RBH - 2013 and beyond stays not in picanet

In [ ]:
dem = pd.read_csv(rbh_dir/"ai_picu_demographics.csv", usecols=["studyID", "PICUadm_dttm"]).drop_duplicates("studyID")
dem["stay_key"] = M.rbh_stay_key(dem["studyID"])
dem["year"] = pd.to_datetime(dem["PICUadm_dttm"], format=M.RBH_TIME_FMT).dt.year
dropped = dem[(dem["year"] >= 2013) & ~dem["stay_key"].isin(set(rbh_m["stay_key"]))].drop_duplicates("stay_key")
print(dropped["year"].value_counts().sort_index().to_string())

## Baseline: manifest waterfalls (before the step 1 and step 2 rebuild)
Reruns the three manifests with the current code and saves each waterfall as a CSV in `derived/reports/baseline/`. After the rebuild we rerun this cell and compare; any difference is a regression. Counts only, no IDs.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import manifest as M

base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
out = open_picu_dir / "derived" / "reports" / "baseline"
out.mkdir(parents=True, exist_ok=True)

rbh_m, rbh_w = M.rbh_manifest(base / "RBH")
smh_m, smh_w = M.smh_manifest(base / "SMH")
gosh_m, gosh_w = M.gosh_manifest(base / "GOSH")

for site, w, m in [("RBH", rbh_w, rbh_m), ("SMH", smh_w, smh_m), ("GOSH", gosh_w, gosh_m)]:
    M.print_waterfall(w, site)
    pd.DataFrame(w).to_csv(out / f"{site}_waterfall.csv", index=False)
    print(site, "manifest stays:", len(m), "\n")